# FAERS Dataset -- Pediatric vs Adult Characteristics Analysis

วิเคราะห์ลักษณะของรายงาน ICSR ในกลุ่ม **Pediatric** (อายุ < 19 ปี) เทียบกับ **Adult** (>= 19 ปี)  
และแยกกลุ่มอายุตามเกณฑ์ **NICHD** ในกลุ่มเด็ก

---
**แหล่งข้อมูล:**
- `overall_characteristics_summary.csv` -- สรุป Pediatric vs Adult
- `overview_v2.csv` -- สรุปแยกตาม NICHD age groups

| กลุ่ม | N |
|---|---|
| Pediatric | 221,182 |
| Adult | 2,506,379 |

## 0 -- Setup

In [ ]:
import pathlib, warnings, re
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

DATA_DIR = pathlib.Path('output/csv')
df_ov = pd.read_csv(DATA_DIR / 'overall_characteristics_summary.csv')
df_v2 = pd.read_csv(DATA_DIR / 'overview_v2.csv')

# Rename columns
df_ov.columns = ['section','item','Pediatric','Adult']
df_ov['section'] = df_ov['section'].replace('', float('nan')).ffill()
df_ov['item'] = df_ov['item'].fillna('')

df_v2.columns = ['section','item','Infancy','Toddler','Early childhood',
                  'Middle childhood','Early adolescence','Late adolescence','Adult']
df_v2['section'] = df_v2['section'].replace('', float('nan')).ffill()
df_v2['item'] = df_v2['item'].fillna('')

print('df_ov shape:', df_ov.shape)
print('df_v2 shape:', df_v2.shape)


## 1 -- Helper Functions

In [ ]:
def parse_val(s):
    """Extract (count, pct) from '108,522 (49.1%)'"""
    if pd.isna(s) or str(s).strip() == '':
        return (float('nan'), float('nan'))
    m = re.match(r'([\d,]+)\s*\(([\d.]+)%?\)', str(s).strip())
    if m:
        return int(m.group(1).replace(',', '')), float(m.group(2))
    try:
        return int(str(s).replace(',', '')), float('nan')
    except Exception:
        return float('nan'), float('nan')

def get_section(df, section_val):
    return df[df['section'].str.strip() == section_val].copy()

GROUPS = ['Infancy','Toddler','Early childhood',
          'Middle childhood','Early adolescence','Late adolescence','Adult']
GROUPS_PED = GROUPS[:-1]
GROUPS_SHORT = ['Infancy\n(0-2y)','Toddler\n(2-5y)','Early\nChild\n(5-10y)',
                'Mid\nChild\n(10-14y)','Early\nAdol\n(14-17y)','Late\nAdol\n(17-19y)','Adult\n(>=19y)']
GROUPS_SHORT_PED = GROUPS_SHORT[:-1]

n_row = df_v2[df_v2['section'] == 'Number of Feature'].iloc[0]
ns = {g: int(str(n_row[g]).replace(',','')) for g in GROUPS}
print('N per group:')
for g in GROUPS:
    print(f'  {g:<22}: {ns[g]:>10,}')


---
# A -- Overall Characteristics: Pediatric vs Adult

เปรียบเทียบลักษณะทั่วไประหว่างกลุ่ม Pediatric (N=221,182) และ Adult (N=2,506,379)


## A.1 -- Gender Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels_g = ['Female', 'Male', 'Unknown']
colors_g = ['#E87A7A', '#6B9BD2', '#B0B0B0']
sec_g = get_section(df_ov, 'Gender')

for ax, grp, n_label in zip(axes, ['Pediatric','Adult'], ['221,182','2,506,379']):
    vals = [parse_val(sec_g[sec_g['item']==lb][grp].values[0])[1] for lb in labels_g]
    bars = ax.barh(labels_g[::-1], vals[::-1], color=colors_g[::-1], height=0.5, edgecolor='white')
    for bar, v in zip(bars, vals[::-1]):
        ax.text(v+0.5, bar.get_y()+bar.get_height()/2, f'{v:.1f}%',
                va='center', ha='left', fontsize=10, fontweight='bold')
    ax.set_xlim(0, 75)
    ax.set_xlabel('Percentage (%)')
    ax.axvline(50, color='gray', lw=0.8, ls='--', alpha=0.4)
    ax.set_title(f'{grp}\n(n = {n_label})', fontweight='bold')

plt.suptitle('Gender Distribution -- Pediatric vs Adult', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/figures/fig_A1_gender.png', bbox_inches='tight')
plt.show()


### การวิเคราะห์ -- เพศ

- **Pediatric**: เพศใกล้เคียงกันมาก (Female 49.1% / Male 47.4%) สะท้อนสัดส่วนประชากรเด็กทั่วไป
- **Adult**: เพศหญิงมีสัดส่วนสูงกว่าชัดเจน (Female **56.8%** vs Male 41.5%)  
  เป็นรูปแบบที่พบบ่อยใน pharmacovigilance databases เนื่องจากผู้หญิงรายงาน ADR มากกว่า
- **Unknown** ใน Pediatric สูงกว่า (3.5% vs 1.7%) -- การรายงานข้อมูลเด็กมักไม่สมบูรณ์


## A.2 -- Reporter Country

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
sec = get_section(df_ov, 'Reporter country')
items_c = ['United States', 'Foreign']
ped_v = [parse_val(sec[sec['item']==it]['Pediatric'].values[0])[1] for it in items_c]
adu_v = [parse_val(sec[sec['item']==it]['Adult'].values[0])[1] for it in items_c]
y = np.arange(len(items_c)); h = 0.3
b1 = ax.barh(y+h/2, ped_v, height=h, color='#4E79A7', label='Pediatric')
b2 = ax.barh(y-h/2, adu_v, height=h, color='#F28E2B', label='Adult')
for bars, vals in [(b1,ped_v),(b2,adu_v)]:
    for bar, v in zip(bars, vals):
        ax.text(v+0.5, bar.get_y()+bar.get_height()/2, f'{v:.1f}%', va='center', ha='left', fontsize=10)
ax.set_yticks(y); ax.set_yticklabels(items_c)
ax.set_xlim(0, 75); ax.axvline(50, color='gray', lw=0.8, ls='--', alpha=0.4)
ax.set_xlabel('Percentage (%)'); ax.legend()
ax.set_title('Reporter Country -- Pediatric vs Adult', fontweight='bold')
plt.tight_layout()
plt.savefig('output/figures/fig_A2_country.png', bbox_inches='tight')
plt.show()


### การวิเคราะห์ -- ประเทศผู้รายงาน

- ทั้งสองกลุ่มมีสัดส่วน US / Foreign ใกล้เคียงกันมาก (~50:50)
- Pediatric: Foreign **53.1%** / US 46.9%
- Adult: Foreign 51.1% / US 48.9%
- ชี้ให้เห็นว่าฐาน FAERS ได้รับรายงานจากทั่วโลก ไม่ใช่ข้อมูลสหรัฐฯ เพียงอย่างเดียว


## A.3 -- Reporter Qualification

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sec = get_section(df_ov, 'Reporter qualification')
items_q = ['Physician', 'Pharmacist', 'Other health professional']
ped_v = [parse_val(sec[sec['item']==it]['Pediatric'].values[0])[1] for it in items_q]
adu_v = [parse_val(sec[sec['item']==it]['Adult'].values[0])[1] for it in items_q]
y = np.arange(len(items_q)); h = 0.3
b1 = ax.barh(y+h/2, ped_v, height=h, color='#4E79A7', label='Pediatric')
b2 = ax.barh(y-h/2, adu_v, height=h, color='#F28E2B', label='Adult')
for bars, vals in [(b1,ped_v),(b2,adu_v)]:
    for bar, v in zip(bars, vals):
        ax.text(v+0.5, bar.get_y()+bar.get_height()/2, f'{v:.1f}%', va='center', ha='left', fontsize=10)
ax.set_yticks(y); ax.set_yticklabels(items_q)
ax.set_xlim(0, 62); ax.set_xlabel('Percentage (%)')
ax.legend(); ax.set_title('Reporter Qualification -- Pediatric vs Adult', fontweight='bold')
plt.tight_layout()
plt.savefig('output/figures/fig_A3_qualification.png', bbox_inches='tight')
plt.show()
ratio = adu_v[items_q.index('Pharmacist')] / ped_v[items_q.index('Pharmacist')]
print(f'Pharmacist: Adult {adu_v[1]:.1f}% vs Ped {ped_v[1]:.1f}% -- ratio {ratio:.1f}x')


### การวิเคราะห์ -- คุณสมบัติผู้รายงาน

- **Pharmacist รายงาน ADR ใน Adult มากกว่าเด็กถึง ~1.8 เท่า** (15.0% vs 8.4%)  
  สะท้อนบทบาทของเภสัชกรในการดูแลผู้ป่วยผู้ใหญ่ที่มีโรคเรื้อรังและใช้ยาหลายชนิด
- **Physician** สัดส่วนสูงกว่าในกลุ่ม Pediatric (45.4% vs 42.4%)  
  การรักษาเด็กมักผ่านกุมารแพทย์โดยตรง ไม่ผ่านเภสัชกร
- **Other health professional** สัดส่วนใกล้เคียงกัน (~42-46%) ทั้งสองกลุ่ม


## A.4 -- Serious Outcomes

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
sec = get_section(df_ov, 'Serious outcome reported')
items_s = ['Death','Hospitalization','Life threatening',
           'Disability','Congenital anomaly','Other serious','Any serious outcome']
ped_v = []; adu_v = []
for it in items_s:
    r = sec[sec['item']==it]
    if len(r):
        ped_v.append(parse_val(r['Pediatric'].values[0])[1])
        adu_v.append(parse_val(r['Adult'].values[0])[1])
    else:
        ped_v.append(float('nan')); adu_v.append(float('nan'))
y = np.arange(len(items_s)); h = 0.3
b1 = ax.barh(y+h/2, ped_v, height=h, color='#4E79A7', label='Pediatric')
b2 = ax.barh(y-h/2, adu_v, height=h, color='#F28E2B', label='Adult')
for bars, vals in [(b1,ped_v),(b2,adu_v)]:
    for bar, v in zip(bars, vals):
        if not (isinstance(v, float) and v != v):
            ax.text(v+0.3, bar.get_y()+bar.get_height()/2, f'{v:.1f}%', va='center', ha='left', fontsize=9)
ax.set_yticks(y); ax.set_yticklabels(items_s)
ax.set_xlim(0, 95); ax.axvline(50, color='gray', lw=0.8, ls='--', alpha=0.4)
ax.set_xlabel('Percentage (% of ICRSs)')
ax.legend(); ax.set_title('Serious Outcomes -- Pediatric vs Adult\n(% of total ICRSs in each group)', fontweight='bold')
plt.tight_layout()
plt.savefig('output/figures/fig_A4_serious.png', bbox_inches='tight')
plt.show()
print('Delta (Ped - Adult):')
for it, p, a in zip(items_s, ped_v, adu_v):
    if not (isinstance(p, float) and p != p):
        sign = 'Ped>' if p > a else 'Adult>'
        print(f'  {it:<25}: Ped {p:.1f}%  Adult {a:.1f}%  [{sign} by {abs(p-a):.1f}pp]')


### การวิเคราะห์ -- Serious Outcomes

| Outcome | Pediatric | Adult | ความแตกต่าง |
|---|---|---|---|
| **Any serious** | **75.1%** | **75.1%** | เท่ากันพอดี |
| Death | 6.9% | **12.9%** | Adult สูงกว่า ~2x |
| Hospitalization | 35.4% | 35.6% | ใกล้เคียงกัน |
| Life-threatening | **7.3%** | 5.7% | Ped สูงกว่าเล็กน้อย |
| Congenital anomaly | **1.3%** | 0.1% | Ped สูงกว่า **13x** |
| Disability | 1.5% | 2.1% | Adult สูงกว่าเล็กน้อย |

**ข้อสังเกตสำคัญ:**
1. **Any serious outcome เท่ากันพอดี 75.1%** -- selection bias ของ FAERS ต่อ serious cases เท่ากันทั้งสองกลุ่ม
2. **Death ใน Adult เกือบ 2 เท่า** (12.9% vs 6.9%) -- โรคร่วมและความเปราะบางในผู้ใหญ่
3. **Congenital anomaly ในเด็กสูงกว่า 13 เท่า** -- ทารกที่ได้รับผลจากยาขณะอยู่ในครรภ์
4. **Life-threatening ในเด็กสูงกว่าเล็กน้อย** (7.3% vs 5.7%) -- ADR เฉียบพลันในเด็ก


## A.5 -- Products and Events per ICSR (Polypharmacy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
configs = [
    ('Suspect product reported per ICSR', 'Suspect Products per ICSR'),
    ('Adverse event reported per ICSR',   'Adverse Events per ICSR'),
]
for ax, (sec_name, title) in zip(axes, configs):
    sec = get_section(df_ov, sec_name)
    ped_v = [parse_val(sec[sec['item']==it]['Pediatric'].values[0])[1] for it in ['One','Multiple']]
    adu_v = [parse_val(sec[sec['item']==it]['Adult'].values[0])[1] for it in ['One','Multiple']]
    x = np.arange(2); w = 0.35
    b1 = ax.bar(x-w/2, ped_v, width=w, color='#4E79A7', label='Pediatric', edgecolor='white')
    b2 = ax.bar(x+w/2, adu_v, width=w, color='#F28E2B', label='Adult',     edgecolor='white')
    for bars, vals in [(b1,ped_v),(b2,adu_v)]:
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                    f'{v:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(['One','Multiple'])
    ax.set_ylim(0, 90); ax.set_ylabel('Percentage (%)')
    ax.set_title(title, fontweight='bold'); ax.legend()
plt.suptitle('Number of Reported Items per ICSR', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/figures/fig_A5_icsr.png', bbox_inches='tight')
plt.show()


### การวิเคราะห์ -- Polypharmacy และ Multi-ADR

- **รูปแบบใกล้เคียงกันมากระหว่างเด็กและผู้ใหญ่**
- ~**31% ของรายงาน** มียาต้องสงสัยมากกว่า 1 ชนิด (polypharmacy) ทั้งสองกลุ่ม
- ~**45% ของรายงาน** มี ADR มากกว่า 1 ชนิดในรายงานเดียวกัน
- ความซับซ้อนของรายงาน (polypharmacy, multiple ADRs) ไม่แตกต่างอย่างมีนัยสำคัญระหว่างเด็กและผู้ใหญ่


---
# B -- NICHD Age Group Analysis (Pediatric Only)

แบ่งกลุ่มเด็กตาม **NICHD developmental stage** เพื่อดูแนวโน้มตามอายุ

| กลุ่ม | ช่วงอายุ |
|---|---|
| Infancy | 0-2 ปี |
| Toddler | 2-5 ปี |
| Early childhood | 5-10 ปี |
| Middle childhood | 10-14 ปี |
| Early adolescence | 14-17 ปี |
| Late adolescence | 17-19 ปี |


## B.1 -- NICHD Group Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
total_ped = sum(ns[g] for g in GROUPS_PED)
ped_ns  = [ns[g] for g in GROUPS_PED]
ped_pct = [n/total_ped*100 for n in ped_ns]
colors_b = plt.cm.Blues(np.linspace(0.4, 0.9, len(GROUPS_PED)))
bars = ax.bar(GROUPS_SHORT_PED, ped_ns, color=colors_b, edgecolor='white', linewidth=1.2)
for bar, n, pct in zip(bars, ped_ns, ped_pct):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
            f'{n:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax.set_ylabel('Number of ICRSs')
ax.set_title(f'NICHD Age Group Distribution\nTotal Pediatric = {total_ped:,}', fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, max(ped_ns)*1.25)
plt.tight_layout()
plt.savefig('output/figures/fig_B1_nichd_dist.png', bbox_inches='tight')
plt.show()
adol = ns['Early adolescence']+ns['Late adolescence']
print(f'Adolescents (14-19y): {adol:,} ({adol/total_ped*100:.1f}% of Pediatric)')


### การวิเคราะห์ -- การกระจายตัว NICHD

- **Early adolescence (14-17y)** มีรายงานมากที่สุด -- **72,516 (32.8%)**
- **Late adolescence (17-19y)** อันดับสอง -- **53,992 (24.4%)**
- **วัยรุ่น (14-19y) รวมกัน 57.2%** ของรายงาน Pediatric ทั้งหมด
- **Toddler (2-5y)** มีน้อยที่สุด -- 8,269 (3.7%)
- แนวโน้มเพิ่มขึ้นตามอายุ สะท้อนการใช้ยาที่มากขึ้นในวัยรุ่น


## B.2 -- Gender Trend Across NICHD Groups (Key Finding)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sec_g2 = df_v2[df_v2['section'] == 'Gender'].copy()
female_p = [parse_val(sec_g2[sec_g2['item']=='Female'][g].values[0])[1] for g in GROUPS]
male_p   = [parse_val(sec_g2[sec_g2['item']=='Male'][g].values[0])[1] for g in GROUPS]
x = np.arange(len(GROUPS))
ax.plot(x, female_p, 'o-', color='#E87A7A', lw=2.5, ms=9, label='Female', zorder=3)
ax.plot(x, male_p,   's-', color='#6B9BD2', lw=2.5, ms=9, label='Male',   zorder=3)
for i, (f, m) in enumerate(zip(female_p, male_p)):
    ax.annotate(f'{f:.1f}%', (i,f), xytext=(0,8), textcoords='offset points',
                ha='center', fontsize=9, color='#C0392B', fontweight='bold')
    ax.annotate(f'{m:.1f}%', (i,m), xytext=(0,-15), textcoords='offset points',
                ha='center', fontsize=9, color='#2471A3', fontweight='bold')
ax.axvline(4, color='green', lw=2, ls='--', alpha=0.7)
ax.text(4.08, 63, 'Gender flip\n(14-17y)', color='green', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#EAFAF1', alpha=0.8))
ax.axhline(50, color='gray', lw=0.8, ls=':', alpha=0.5)
ax.fill_between(x[:5], female_p[:5], male_p[:5], alpha=0.06, color='#6B9BD2')
ax.fill_between(x[4:], female_p[4:], male_p[4:], alpha=0.06, color='#E87A7A')
ax.set_xticks(x); ax.set_xticklabels(GROUPS_SHORT)
ax.set_ylim(30, 72); ax.set_ylabel('Percentage (%)')
ax.legend(loc='upper left')
ax.set_title('Gender Distribution Across NICHD Age Groups\n*** Gender flips at Early Adolescence (14-17y) ***',
             fontweight='bold')
plt.tight_layout()
plt.savefig('output/figures/fig_B2_gender_trend.png', bbox_inches='tight')
plt.show()


### การวิเคราะห์ -- Gender Flip (การค้นพบสำคัญ)

**การพลิกสัดส่วนเพศที่ช่วงอายุ 14 ปี เป็นข้อค้นพบทางชีววิทยาที่สำคัญที่สุด:**

| กลุ่มอายุ | Female | Male | ผู้นำ |
|---|---|---|---|
| Infancy (0-2y) | 39.5% | 51.8% | Male |
| Toddler (2-5y) | 42.2% | 53.1% | Male |
| Early childhood (5-10y) | 41.7% | 53.3% | Male |
| Middle childhood (10-14y) | 42.7% | 54.0% | Male |
| **Early adolescence (14-17y)** | **52.2%** | 44.9% | **Female** |
| Late adolescence (17-19y) | 57.6% | 40.4% | Female |
| Adult (>=19y) | 56.8% | 41.5% | Female |

**การตีความ:**
- เด็กชายมีสัดส่วนสูงกว่าในวัยเด็กเล็ก (51-54%) -- สอดคล้องกับโรคที่พบในเด็กชายมากกว่า เช่น ADHD, autism
- จุดเปลี่ยนที่ **Early adolescence (14-17y)** ตรงกับช่วงวัยเจริญพันธุ์
- เพศหญิงเริ่มใช้ยามากขึ้นจากยาคุมกำเนิด, ยารักษาซึมเศร้า/วิตกกังวล, ยาสิว
- ฮอร์โมนเพศมีผลต่อ drug metabolism และ ADR susceptibility อย่างมีนัยสำคัญ


## B.3 -- Death Rate by NICHD Group (U-shaped Pattern)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sec_s = df_v2[df_v2['section'] == 'Serious outcome reported'].copy()
death_p = [parse_val(sec_s[sec_s['item']=='Death'][g].values[0])[1] for g in GROUPS]
death_n = [parse_val(sec_s[sec_s['item']=='Death'][g].values[0])[0] for g in GROUPS]
bar_c = ['#E74C3C' if v > 10 else '#E59866' if v > 6 else '#FAD7A0' for v in death_p]
bars = ax.bar(GROUPS_SHORT, death_p, color=bar_c, edgecolor='white', linewidth=1.2)
for bar, v, n in zip(bars, death_p, death_n):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{v:.1f}%\n(n={n:,})', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Death Rate (% of ICRSs in group)')
ax.set_title('Death Rate Across NICHD Age Groups (U-shaped Pattern)', fontweight='bold')
ax.set_ylim(0, 20)
# U-shape curve
from matplotlib.patches import FancyArrowPatch
ax.annotate('U-shaped\npattern', xy=(3, death_p[3]), xytext=(3, 9),
            arrowprops=dict(arrowstyle='->', color='#922B21', lw=1.5),
            ha='center', fontsize=10, color='#922B21',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#FADBD8', alpha=0.7))
plt.tight_layout()
plt.savefig('output/figures/fig_B3_death.png', bbox_inches='tight')
plt.show()


### การวิเคราะห์ -- Death Rate แบบ U-shaped

| กลุ่ม | Death Rate |
|---|---|
| **Infancy (0-2y)** | **13.9%** -- สูงสุดในกลุ่มเด็ก |
| Toddler (2-5y) | 11.6% |
| Early childhood (5-10y) | 7.8% |
| **Middle childhood (10-14y)** | **5.5%** -- ต่ำสุด |
| Early adolescence (14-17y) | 5.5% |
| Late adolescence (17-19y) | 6.5% |
| **Adult (>=19y)** | **12.9%** -- กลับสูงอีกครั้ง |

- **Infancy สูงสุด** -- ทารกมีอวัยวะยังพัฒนาไม่สมบูรณ์ และ drug metabolism จำกัด
- **Middle/Early adolescence ต่ำสุด** -- ร่างกายแข็งแรงที่สุด
- **Adult กลับสูงขึ้น** -- ภาระโรคสะสม โรคร่วม และความเปราะบางที่เพิ่มขึ้น


## B.4 -- Congenital Anomaly by NICHD Group

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
sec_s = df_v2[df_v2['section'] == 'Serious outcome reported'].copy()
ca_p = [parse_val(sec_s[sec_s['item']=='Congenital anomaly'][g].values[0])[1] for g in GROUPS]
ca_n = [parse_val(sec_s[sec_s['item']=='Congenital anomaly'][g].values[0])[0] for g in GROUPS]
bar_c = ['#1ABC9C' if i == 0 else '#BDC3C7' for i in range(len(GROUPS))]
bars = ax.bar(GROUPS_SHORT, ca_p, color=bar_c, edgecolor='white', linewidth=1.2)
for bar, v, n in zip(bars, ca_p, ca_n):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f'{v:.1f}%\n(n={n:,})', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Congenital Anomaly Rate (%)')
ax.set_title('Congenital Anomaly Rate Across NICHD Age Groups', fontweight='bold')
ax.set_ylim(0, 22)
ax.text(0, 17.5, '15.0% in Infancy\n(~150x higher than Adult 0.1%)',
        ha='center', fontsize=9, color='#0E6655',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#D5F5E3', alpha=0.8))
plt.tight_layout()
plt.savefig('output/figures/fig_B4_congenital.png', bbox_inches='tight')
plt.show()


### การวิเคราะห์ -- Congenital Anomaly

- **Infancy มี congenital anomaly rate สูงผิดปกติ (15.0%)** ในขณะที่กลุ่มอื่นต่ำกว่า 1.2%
- สัดส่วนใน Infancy สูงกว่า Adult ถึง **~150 เท่า** (15.0% vs 0.1%)
- Toddler ลดลงอย่างรวดเร็วเหลือเพียง 1.1%
- **รายงานใน Infancy ส่วนใหญ่เกี่ยวข้องกับ:**
  1. ทารกที่ได้รับยาผ่านมารดาขณะตั้งครรภ์ (in utero exposure)
  2. ทารกที่คลอดมาพร้อม birth defects ที่เชื่อมโยงกับยา (drug-induced congenital defects)
  3. Neonatal withdrawal syndrome


## B.5 -- Reporter Country Trend Across NICHD Groups

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sec_c2 = df_v2[df_v2['section'] == 'Reporter country'].copy()
foreign_p = [parse_val(sec_c2[sec_c2['item']=='Foreign'][g].values[0])[1] for g in GROUPS]
us_p      = [parse_val(sec_c2[sec_c2['item']=='United States'][g].values[0])[1] for g in GROUPS]
x = np.arange(len(GROUPS))
ax.plot(x, foreign_p, 'o-', color='#2980B9', lw=2.5, ms=8, label='Foreign')
ax.plot(x, us_p,      's-', color='#E74C3C', lw=2.5, ms=8, label='United States')
for i, (f, u) in enumerate(zip(foreign_p, us_p)):
    ax.annotate(f'{f:.1f}%', (i,f), xytext=(0,7), textcoords='offset points',
                ha='center', fontsize=9, color='#1A5276', fontweight='bold')
    ax.annotate(f'{u:.1f}%', (i,u), xytext=(0,-15), textcoords='offset points',
                ha='center', fontsize=9, color='#922B21', fontweight='bold')
ax.axhline(50, color='gray', lw=0.8, ls=':', alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(GROUPS_SHORT)
ax.set_ylim(20, 85); ax.set_ylabel('Percentage (%)')
ax.legend(); ax.set_title('Reporter Country Trend Across NICHD Age Groups', fontweight='bold')
plt.tight_layout()
plt.savefig('output/figures/fig_B5_country.png', bbox_inches='tight')
plt.show()


### การวิเคราะห์ -- ประเทศผู้รายงานตามกลุ่มอายุ

- **Infancy มีสัดส่วนรายงานจากต่างประเทศสูงที่สุด (68.1%)** -- โดดเด่นมากเทียบกับกลุ่มอื่น
- สัดส่วน US เพิ่มขึ้นตามอายุ: 31.9% (Infancy) -> 50.9% (Late adolescence)
- อาจสะท้อนว่าประเทศนอก US มีการเฝ้าระวัง ADR ในทารกที่เข้มงวดกว่า
- หรือมี specific ADR clusters ในทารก (เช่น neonatal drug reactions) ที่รายงานจากยุโรป/เอเชียมาก


## B.6 -- Heatmap: All Serious Outcomes by NICHD Group

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
outcomes_h = ['Death','Hospitalization','Life threatening','Disability','Congenital anomaly','Other serious']
sec_s2 = df_v2[df_v2['section'] == 'Serious outcome reported'].copy()
matrix = []
for out in outcomes_h:
    row_d = [parse_val(sec_s2[sec_s2['item']==out][g].values[0])[1] for g in GROUPS]
    matrix.append(row_d)
mat = np.array(matrix, dtype=float)
im = ax.imshow(mat, cmap='YlOrRd', aspect='auto', vmin=0, vmax=60)
plt.colorbar(im, ax=ax, label='%', shrink=0.8)
ax.set_xticks(range(len(GROUPS))); ax.set_xticklabels(GROUPS_SHORT, fontsize=9.5)
ax.set_yticks(range(len(outcomes_h))); ax.set_yticklabels(outcomes_h)
for i in range(len(outcomes_h)):
    for j in range(len(GROUPS)):
        v = mat[i, j]
        col = 'white' if v > 35 else 'black'
        ax.text(j, i, f'{v:.1f}%', ha='center', va='center', fontsize=9, color=col, fontweight='bold')
ax.set_title('Serious Outcome Rates (%) by NICHD Age Group\n(darker = higher rate)', fontweight='bold')
plt.tight_layout()
plt.savefig('output/figures/fig_B6_heatmap.png', bbox_inches='tight')
plt.show()


### การวิเคราะห์ -- Heatmap Serious Outcomes

1. **Death**: รูปแบบ U-shaped ชัดเจน -- Infancy (13.9%) และ Adult (12.9%) สูง; Middle childhood (5.5%) ต่ำสุด
2. **Hospitalization**: สูงสม่ำเสมอทุกกลุ่ม (~31-41%) -- Infancy/Toddler สูงกว่าเล็กน้อย
3. **Life threatening**: ลดลงตามอายุจาก Infancy (9.3%) -> Middle childhood (5.8%)
4. **Congenital anomaly**: **เฉพาะ Infancy เท่านั้น (15.0%)** ทุกกลุ่มอื่นต่ำมาก (<1.2%)
5. **Other serious**: ลดลงตามอายุ (Infancy 59.4% -> Late adol. 48.7%) แต่ Adult ขึ้นมา (46.8%)
6. **Non-serious**: เพิ่มขึ้นตามอายุ -- Infancy (8.6%) -> Late adol. (28.6%) -> Adult (24.9%)


---
# C -- Summary & Key Findings

## สรุปผลการวิเคราะห์ทั้งหมด

| หัวข้อ | ผลการวิเคราะห์ |
|---|---|
| **ขนาดข้อมูล** | Adult มากกว่า Ped ถึง **11.3 เท่า** (2.5M vs 221K ICSRs) |
| **Gender (Ped)** | ใกล้เคียงกัน: Female 49.1% / Male 47.4% |
| **Gender (Adult)** | Female dominant: 56.8% / Male 41.5% |
| **Gender flip** | เพศสลับที่ **Early adolescence (14-17y)** -- สอดคล้องกับ puberty |
| **Any serious outcome** | **75.1% เท่ากันทั้งสองกลุ่ม** -- selection bias ของ FAERS |
| **Death (Pediatric)** | U-shaped: Infancy (13.9%) -> ต่ำสุด Middle childhood (5.5%) |
| **Death (Adult)** | 12.9% -- ใกล้เคียงกับ Infancy |
| **Congenital anomaly** | Infancy สูง **15.0%** (Adult เพียง 0.1%) |
| **Polypharmacy** | ~31% ของทุกกลุ่มมียาต้องสงสัย > 1 ชนิด |
| **Multi-ADR** | ~39-46% ของทุกกลุ่มมี ADR > 1 ชนิด |
| **Pharmacist reporter** | Adult 15.0% vs Ped 8.4% -- ต่างกัน ~1.8 เท่า |
| **Adolescent dominance** | Early + Late adolescence = **57.2%** ของ Ped |
| **Foreign reports (Infancy)** | 68.1% -- สูงผิดปกติเทียบกับกลุ่มอื่น (~50%) |

---
*Analysis based on FDA FAERS deduplicated data (1 row per safetyreportid)*  
*Pediatric: age < 19 years | Adult: age >= 19 years*


## C.1 -- Summary 4-Panel Figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Gender trend
ax = axes[0,0]
sec_g2 = df_v2[df_v2['section']=='Gender'].copy()
fp = [parse_val(sec_g2[sec_g2['item']=='Female'][g].values[0])[1] for g in GROUPS]
mp = [parse_val(sec_g2[sec_g2['item']=='Male'][g].values[0])[1] for g in GROUPS]
x = np.arange(len(GROUPS))
ax.plot(x, fp, 'o-', color='#E87A7A', lw=2.5, ms=7, label='Female')
ax.plot(x, mp, 's-', color='#6B9BD2', lw=2.5, ms=7, label='Male')
ax.axhline(50, color='gray', lw=0.8, ls=':', alpha=0.5)
ax.axvline(4, color='green', lw=1.5, ls='--', alpha=0.6)
ax.text(4.05, 63, 'Gender flip', color='green', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(GROUPS_SHORT, fontsize=7)
ax.set_ylim(30, 70); ax.set_ylabel('%'); ax.legend(fontsize=9)
ax.set_title('Gender Trend by NICHD', fontweight='bold')

# Panel 2: Death U-shape
ax = axes[0,1]
sec_s3 = df_v2[df_v2['section']=='Serious outcome reported'].copy()
dp = [parse_val(sec_s3[sec_s3['item']=='Death'][g].values[0])[1] for g in GROUPS]
bc = ['#E74C3C' if v > 10 else '#E59866' if v > 6 else '#FAD7A0' for v in dp]
ax.bar(GROUPS_SHORT, dp, color=bc, edgecolor='white')
for i, v in enumerate(dp):
    ax.text(i, v+0.1, f'{v:.1f}%', ha='center', fontsize=8.5, fontweight='bold')
ax.set_ylabel('%'); ax.set_ylim(0, 18)
ax.set_title('Death Rate (U-shaped)', fontweight='bold')

# Panel 3: Congenital anomaly
ax = axes[1,0]
cp = [parse_val(sec_s3[sec_s3['item']=='Congenital anomaly'][g].values[0])[1] for g in GROUPS]
bc2 = ['#1ABC9C' if i==0 else '#BDC3C7' for i in range(len(GROUPS))]
ax.bar(GROUPS_SHORT, cp, color=bc2, edgecolor='white')
for i, v in enumerate(cp):
    ax.text(i, v+0.1, f'{v:.1f}%', ha='center', fontsize=8.5, fontweight='bold')
ax.set_ylabel('%'); ax.set_ylim(0, 20)
ax.set_title('Congenital Anomaly Rate', fontweight='bold')

# Panel 4: NICHD distribution
ax = axes[1,1]
pn = [ns[g] for g in GROUPS_PED]
pp = [n/sum(pn)*100 for n in pn]
bc3 = plt.cm.Blues(np.linspace(0.4, 0.9, len(GROUPS_PED)))
bars = ax.bar(GROUPS_SHORT_PED, pp, color=bc3, edgecolor='white')
for bar, p in zip(bars, pp):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{p:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('%'); ax.set_ylim(0, 40)
ax.set_title('NICHD Group Distribution', fontweight='bold')

plt.suptitle('FAERS Analysis -- Key Findings Summary', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('output/figures/fig_C_summary.png', bbox_inches='tight')
plt.show()
print('All figures saved.')
